### Import Dependencies

In [41]:
from google import genai
import instructor
from pydantic import BaseModel, Field
from qdrant_client import QdrantClient
from google.genai import types

In [2]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

In [3]:
### Mock Example

prompt = """ You are a helpful assistant
Return an answer to the question
Question: what is your name?
"""

In [6]:
import os
gemini_client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

response = gemini_client.models.generate_content(
    contents=[prompt],
    model="gemini-3.1-flash-lite",
)

print(response.text)

I do not have a name. I am a large language model, trained by Google.


In [7]:
response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text='I do not have a name. I am a large language model, trained by Google.',
            thought_signature=b'\x124\n2\x01\x11M2\x0fK\xef~\xe9R\xc3\xf3\xdb\x1d\xf3\xbd\x120\xac\x13O\r\xdfq\xdfQ\xab\xd3\x8f\xd6k/5j\xa7-fP\xc2S\x9d\xceG\xc9=\x16\xb2YZ-'
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-3.1-flash-lite',
  response_id='AX9SaoneGKeFz7IP7t7CmQs',
  sdk_http_response=HttpResponse(
    headers=<dict len=12>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=18,
    prompt_token_count=22,
    prompt_tokens_details=[
      ModalityTokenCount(
        modality=<MediaModality.TEXT: 'TEXT'>,
        token_count=22
      ),
    ],
    total_token_count=40
  )
)

### Add instructor (structured outputs)

In [33]:
client = instructor.from_genai(
    gemini_client,
    model="gemini-3.1-flash-lite"
)

In [29]:
class Answer(BaseModel):
    answer: str = Field(description="The answer to the question")

In [34]:
response = client.create(
    messages=[
        {"role": "user", "content": prompt}
    ],
    response_model=Answer,
)

In [35]:
response

Answer(answer='I do not have a name. I am a large language model, trained by Google.')

In [36]:
class AnswerwithReasoning(BaseModel):
    reasoning: str = Field(description="The reasoning for the answer")
    answer: str = Field(description="The answer to the question")

In [45]:
response, raw_response = client.create_with_completion(
    messages=[
        {"role": "user", "content": prompt}
    ],
    response_model=AnswerwithReasoning,
)

In [46]:
response

AnswerwithReasoning(reasoning='I am an AI assistant and I do not have a personal identity or a name.', answer='I do not have a name. I am a large language model, trained by Google.')

### Rag Pipeline

In [ ]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="   answer to the question")

In [ ]:
gemini_client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
qdrant_client = QdrantClient(url="http://localhost:6333")


def get_embeddings(text, task_type="SEMANTIC_SIMILARITY", model="gemini-embedding-001"):
    result = gemini_client.models.embed_content(
        model=model,
        contents=text,
        config=types.EmbedContentConfig(task_type=task_type)
    )
    return result.embeddings[0].values

def retrieve_data(query, k=5):
    query_embedding = get_embeddings(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )
    retrieved_context_ids=[]
    retrieved_context=[]
    similarity_scores=[]
    retrieved_context_ratings=[]

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):
    formatted_context = ""
    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formatted_context


def build_prompt(query, preprocessed_context):
    prompt = f"""
    You are a helpful assistant that can answer questions about the products in stock.
    You are given a question and a list of context.
    You need to answer the question based on the product descriptions.

    Instructions:
    - Answer the question based on the provided context only.
    - Never use the word "context" in your answer, refer it as the available products.
    - Do not use markdown formatting.

    Context:
    {preprocessed_context}

    Question: 
    {query}
    """
    return prompt

def generate_answer(prompt):
    response, RAW = client.create_with_completion(
    messages=[
        {"role": "user", "content": prompt}
    ],
    response_model=RAGGenerationResponse,
)

    return response

def rag_pipeline(query, topk_k=5):
    retrieved_context = retrieve_data(query, topk_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(query, preprocessed_context)
    answer = generate_answer(prompt)

    final_answer = {
        "data_object": answer,
        "answer": answer.answer,
        "question": query,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }
    return final_answer

In [48]:
output = rag_pipeline("Any headphones?")

In [ ]:
output

{'data_object': RAG_GeneraionResponse(answer='Yes, the available products include several types of headphones:\n\n- Wireless Earbuds with 4-Mic and Wireless Charging Case (ID: B09V1F24X4)\n- 2 Pack iPhone Earbuds Wired Lightning Headphone (ID: B0B5K9NFHV)\n- Wireless Bluetooth V5.0 Neckband Earphones for Sport (ID: B0BC4PGXFK)\n- WeurGhy Wireless Earbuds with Bluetooth 5.3 (ID: B0BD8FXHPB)\n- Kids Headphones with Mic, Foldable Stereo On-Ear Headset (ID: B09XQMRYBJ)'),
 'answer': 'Yes, the available products include several types of headphones:\n\n- Wireless Earbuds with 4-Mic and Wireless Charging Case (ID: B09V1F24X4)\n- 2 Pack iPhone Earbuds Wired Lightning Headphone (ID: B0B5K9NFHV)\n- Wireless Bluetooth V5.0 Neckband Earphones for Sport (ID: B0BC4PGXFK)\n- WeurGhy Wireless Earbuds with Bluetooth 5.3 (ID: B0BD8FXHPB)\n- Kids Headphones with Mic, Foldable Stereo On-Ear Headset (ID: B09XQMRYBJ)',
 'question': 'Any headphones?',
 'retrieved_context_ids': ['B09V1F24X4',
  'B0B5K9NFHV',
